# 💰 Dashboard de Dividendos

Acompanhamento automático dos dividendos do seu portfólio com:
- 📋 Histórico de dividendos recebidos por ativo
- 📅 Calendário mensal de pagamentos
- 📈 Evolução da renda ao longo do tempo
- 🥧 Distribuição de dividendos por ativo
- 🔮 Projeção de renda anual (forward dividend)
- 📊 Dividend Yield de cada ativo

---
**Dependências:**
```bash
pip install yfinance pandas plotly
```

## 📦 Célula 1 — Importações

In [17]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas carregadas!')

✅ Bibliotecas carregadas!


## ⚙️ Célula 2 — Configuração do Portfólio
> Mesmo portfólio do dashboard principal — edite aqui se necessário.

In [18]:
PORTFOLIO = {
    "AAPL":   {"shares": 1.013,   "avg_price": 198.03},
    "ASML":   {"shares": 0.3281,  "avg_price": 640.02},
    "GOOGL":  {"shares": 0.7847,  "avg_price": 273.98},
    "EMBJ":   {"shares": 1,       "avg_price": 46.53},
    "AVGO":   {"shares": 0.3659,  "avg_price": 210.00},
    "ENB":    {"shares": 3,       "avg_price": 45.59},
    "SO":     {"shares": 2.1269,  "avg_price": 94.82},
    "JNJ":    {"shares": 1,       "avg_price": 157.94},
    "CVS":    {"shares": 3,       "avg_price": 74.38},
    "KO":     {"shares": 2,       "avg_price": 72.38},
    "PG":     {"shares": 1.4796,  "avg_price": 150.07},
    "NTST":   {"shares": 4,       "avg_price": 18.17},
    "O":      {"shares": 5,       "avg_price": 57.57},
    "EQIX":   {"shares": 0.2975,  "avg_price": 843.36},
    "PLD":    {"shares": 1,       "avg_price": 99.16},
    "WPC":    {"shares": 2,       "avg_price": 61.95},
    "NU":     {"shares": 16.7574, "avg_price": 14.69},
    "BDORY":  {"shares": 10,      "avg_price": 4.70},
    "JPM":    {"shares": 0.3685,  "avg_price": 312.00},
    "V":      {"shares": 0.8597,  "avg_price": 325.65},
    "CSPX.L": {"shares": 0.2821,  "avg_price": 744.30},
    "VWRA.L": {"shares": 1.094,   "avg_price": 158.98},
    "XGDU.L": {"shares": 4,       "avg_price": 66.07},
}

# Janela de histórico de dividendos
ANOS_HISTORICO = 3   # quantos anos de dividendos buscar

TICKERS = list(PORTFOLIO.keys())
print(f'📋 {len(TICKERS)} ativos carregados.')

📋 23 ativos carregados.


## 🌐 Célula 3 — Download do Histórico de Dividendos

O `yfinance` retorna o histórico completo de dividendos de cada ativo.
Aqui calculamos também o **dividendo que você recebeu** (dividendo × suas cotas).

In [19]:
def get_dividendos(portfolio: dict, anos: int) -> pd.DataFrame:
    """
    Baixa histórico de dividendos de cada ativo e calcula
    o valor recebido com base nas cotas do portfólio.
    """
    data_inicio = datetime.now() - timedelta(days=365 * anos)
    rows = []
    sem_dividendo = []
    print('⏳ Buscando histórico de dividendos...')

    for ticker, info in portfolio.items():
        try:
            stock = yf.Ticker(ticker)
            divs = stock.dividends

            if divs.empty:
                sem_dividendo.append(ticker)
                continue

            # Filtra pelo período
            divs.index = divs.index.tz_localize(None)
            divs = divs[divs.index >= data_inicio]

            if divs.empty:
                sem_dividendo.append(ticker)
                continue

            shares = info['shares']
            for data, valor_div in divs.items():
                rows.append({
                    'Ticker':          ticker,
                    'Data':            data,
                    'Dividendo/Cota':  round(valor_div, 4),
                    'Cotas':           shares,
                    'Recebido ($)':    round(valor_div * shares, 4),
                    'Ano':             data.year,
                    'Mês':             data.month,
                    'Mês/Ano':         data.strftime('%b/%Y'),
                })
            print(f'  ✅ {ticker}: {len(divs)} pagamentos encontrados')

        except Exception as e:
            print(f'  ❌ Erro em {ticker}: {e}')

    print(f'\n  ⚪ Sem dividendos: {", ".join(sem_dividendo)}')
    return pd.DataFrame(rows)


df_divs = get_dividendos(PORTFOLIO, ANOS_HISTORICO)
print(f'\n📊 Total de pagamentos registrados: {len(df_divs)}')
print(f'💰 Total recebido no período: ${df_divs["Recebido ($)"].sum():.2f}')

⏳ Buscando histórico de dividendos...
  ✅ AAPL: 12 pagamentos encontrados
  ✅ ASML: 12 pagamentos encontrados
  ✅ GOOGL: 8 pagamentos encontrados
  ✅ EMBJ: 1 pagamentos encontrados
  ✅ AVGO: 12 pagamentos encontrados
  ✅ ENB: 11 pagamentos encontrados
  ✅ SO: 12 pagamentos encontrados
  ✅ JNJ: 12 pagamentos encontrados
  ✅ CVS: 12 pagamentos encontrados
  ✅ KO: 12 pagamentos encontrados
  ✅ PG: 12 pagamentos encontrados
  ✅ NTST: 12 pagamentos encontrados
  ✅ O: 36 pagamentos encontrados
  ✅ EQIX: 12 pagamentos encontrados
  ✅ PLD: 12 pagamentos encontrados
  ✅ WPC: 12 pagamentos encontrados
  ✅ BDORY: 19 pagamentos encontrados
  ✅ JPM: 12 pagamentos encontrados
  ✅ V: 12 pagamentos encontrados

  ⚪ Sem dividendos: NU, CSPX.L, VWRA.L, XGDU.L

📊 Total de pagamentos registrados: 243
💰 Total recebido no período: $250.35


## 📋 Célula 4 — Resumo por Ativo

In [20]:
# Busca preço atual e dividend yield de cada ativo
def get_yield_info(portfolio: dict, df_divs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    ultimo_ano = datetime.now().year - 1

    for ticker, info in portfolio.items():
        try:
            stock      = yf.Ticker(ticker)
            preco      = stock.fast_info.last_price
            fi         = stock.info

            # Dividendo anual forward (projetado)
            div_forward = fi.get('dividendRate', 0) or 0

            # Calcula o yield diretamente — evita inconsistências do yfinance
            if preco and preco > 0 and div_forward > 0:
                dy_forward = div_forward / preco
            else:
                dy_forward = 0

            # Total recebido por você no último ano
            mask = (
                (df_divs['Ticker'] == ticker) &
                (df_divs['Ano'] == ultimo_ano)
            )
            recebido_ano = df_divs.loc[mask, 'Recebido ($)'].sum()

            # Projeção anual = div_forward × suas cotas
            projecao_anual = div_forward * info['shares']

            rows.append({
                'Ticker':            ticker,
                'Cotas':             info['shares'],
                'Preço Atual ($)':   round(preco, 2),
                'Div. Forward/Cota': round(div_forward, 4),
                'Yield Atual (%)':   round(dy_forward * 100, 2),
                f'Recebido {ultimo_ano} ($)': round(recebido_ano, 4),
                'Projeção Anual ($)': round(projecao_anual, 4),
                'Projeção Mensal ($)': round(projecao_anual / 12, 4),
            })
        except Exception as e:
            print(f'Erro em {ticker}: {e}')

    return pd.DataFrame(rows).sort_values('Projeção Anual ($)', ascending=False)


df_yield = get_yield_info(PORTFOLIO, df_divs)

total_projecao_anual  = df_yield['Projeção Anual ($)'].sum()
total_projecao_mensal = total_projecao_anual / 12

print('=' * 50)
print('       💰 RESUMO DE DIVIDENDOS')
print('=' * 50)
print(f'  📅 Projeção Anual  : ${total_projecao_anual:.2f}')
print(f'  📆 Projeção Mensal : ${total_projecao_mensal:.2f}')
print('=' * 50)

# Tabela estilizada
df_yield.style \
    .map(lambda v: 'color: #00d4aa' if isinstance(v, (int, float)) and v > 0 else '',
         subset=['Projeção Anual ($)', 'Projeção Mensal ($)', 'Yield Atual (%)']) \
    .format({
        'Preço Atual ($)':      '${:.2f}',
        'Div. Forward/Cota':    '${:.4f}',
        'Yield Atual (%)':      '{:.2f}%',
        'Projeção Anual ($)':   '${:.2f}',
        'Projeção Mensal ($)':  '${:.2f}',
    })

       💰 RESUMO DE DIVIDENDOS
  📅 Projeção Anual  : $86.61
  📆 Projeção Mensal : $7.22


,Ticker,Cotas,Preço Atual ($),Div. Forward/Cota,Yield Atual (%),Recebido 2025 ($),Projeção Anual ($),Projeção Mensal ($)
12,O,5.000000,$64.85,$3.2400,5.00%,17.450000,$16.20,$1.35
5,ENB,3.000000,$53.99,$2.8400,5.26%,6.099000,$8.52,$0.71
8,CVS,3.000000,$76.20,$2.6600,3.49%,7.980000,$7.98,$0.67
15,WPC,2.000000,$71.84,$3.6800,5.12%,7.240000,$7.36,$0.61
6,SO,2.126900,$98.15,$2.9600,3.02%,6.253100,$6.30,$0.52
10,PG,1.479600,$151.29,$4.2300,2.80%,6.181700,$6.26,$0.52
13,EQIX,0.297500,$974.72,$19.2300,1.97%,5.581200,$5.72,$0.48
7,JNJ,1.000000,$244.22,$5.2000,2.13%,5.140000,$5.20,$0.43
14,PLD,1.000000,$131.60,$4.2800,3.25%,4.040000,$4.28,$0.36
9,KO,2.000000,$77.90,$2.0600,2.64%,4.080000,$4.12,$0.34


## 📅 Célula 5 — Calendário Mensal de Dividendos

In [21]:
# Agrupa dividendos recebidos por mês
mensal = (
    df_divs.groupby(['Ano', 'Mês', 'Mês/Ano'])['Recebido ($)']
    .sum()
    .reset_index()
    .sort_values(['Ano', 'Mês'])
)

fig = go.Figure(go.Bar(
    x=mensal['Mês/Ano'],
    y=mensal['Recebido ($)'],
    marker_color='#00d4aa',
    text=[f'${v:.2f}' for v in mensal['Recebido ($)']],
    textposition='outside',
))

fig.update_layout(
    title='💵 Dividendos Recebidos por Mês',
    xaxis_title='Mês',
    yaxis_title='Total Recebido ($)',
    yaxis_tickprefix='$',
    template='plotly_dark',
    height=420,
)
fig.show()

## 🥧 Célula 6 — Distribuição por Ativo

In [22]:
por_ativo = (
    df_divs.groupby('Ticker')['Recebido ($)']
    .sum()
    .reset_index()
    .sort_values('Recebido ($)', ascending=False)
)

fig = px.pie(
    por_ativo,
    values='Recebido ($)',
    names='Ticker',
    title=f'Distribuição de Dividendos por Ativo (últimos {ANOS_HISTORICO} anos)',
    hole=0.45,
    color_discrete_sequence=px.colors.sequential.Teal,
)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.update_layout(template='plotly_dark', height=480)
fig.show()

## 📊 Célula 7 — Dividend Yield por Ativo

In [23]:
df_yield_plot = df_yield[df_yield['Yield Atual (%)'] > 0].sort_values('Yield Atual (%)')

fig = go.Figure(go.Bar(
    x=df_yield_plot['Yield Atual (%)'],
    y=df_yield_plot['Ticker'],
    orientation='h',
    marker_color='#64ffda',
    text=[f"{v:.2f}%" for v in df_yield_plot['Yield Atual (%)']],
    textposition='outside',
))

fig.update_layout(
    title='📈 Dividend Yield Atual por Ativo',
    xaxis_title='Yield (%)',
    xaxis_ticksuffix='%',
    template='plotly_dark',
    height=500,
)
fig.show()

## 🔮 Célula 8 — Projeção de Renda Futura

Simula quanto você receberá em dividendos ao longo dos próximos anos,
assumindo que manterá as mesmas cotas e o mesmo dividend rate atual.

In [24]:
anos_projecao = 10
ano_atual = datetime.now().year

anos   = list(range(ano_atual, ano_atual + anos_projecao + 1))
renda  = [total_projecao_anual] * len(anos)  # sem crescimento
renda_crescimento = [                         # com crescimento de 5% a.a.
    total_projecao_anual * (1.05 ** i)
    for i in range(len(anos))
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=anos,
    y=renda,
    name='Sem crescimento',
    marker_color='#8892b0',
))

fig.add_trace(go.Scatter(
    x=anos,
    y=renda_crescimento,
    name='+5% a.a. (reinvestimento)',
    mode='lines+markers',
    line=dict(color='#00d4aa', width=2.5),
    marker=dict(size=7),
))

fig.update_layout(
    title=f'🔮 Projeção de Renda com Dividendos — {anos_projecao} anos',
    xaxis_title='Ano',
    yaxis_title='Dividendos Anuais ($)',
    yaxis_tickprefix='$',
    template='plotly_dark',
    height=420,
    legend=dict(x=0.02, y=0.98),
)
fig.show()

print(f'\n📌 Renda atual projetada: ${total_projecao_anual:.2f}/ano  |  ${total_projecao_mensal:.2f}/mês')
print(f'📌 Em {anos_projecao} anos (+5%/a.a.):   ${renda_crescimento[-1]:.2f}/ano  |  ${renda_crescimento[-1]/12:.2f}/mês')


📌 Renda atual projetada: $86.61/ano  |  $7.22/mês
📌 Em 10 anos (+5%/a.a.):   $141.08/ano  |  $11.76/mês


## 📆 Célula 9 — Histórico de Dividendos por Ativo (detalhado)

In [25]:
# Evolução do dividendo/cota ao longo do tempo por ativo
ativos_com_div = df_divs['Ticker'].unique()

fig = go.Figure()
for ticker in ativos_com_div:
    sub = df_divs[df_divs['Ticker'] == ticker].sort_values('Data')
    fig.add_trace(go.Scatter(
        x=sub['Data'],
        y=sub['Dividendo/Cota'],
        mode='lines+markers',
        name=ticker,
        marker=dict(size=5),
    ))

fig.update_layout(
    title='Histórico do Dividendo por Cota',
    xaxis_title='Data',
    yaxis_title='Dividendo/Cota ($)',
    yaxis_tickprefix='$',
    template='plotly_dark',
    hovermode='x unified',
    height=480,
)
fig.show()

## 💾 Célula 10 — Exportar para CSV

In [26]:
#timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Histórico completo
#arq_historico = f'dividendos_historico_{timestamp}.csv'
#df_divs.to_csv(arq_historico, index=False, sep=';', decimal=',')

# Resumo por ativo
#arq_resumo = f'dividendos_resumo_{timestamp}.csv'
#df_yield.to_csv(arq_resumo, index=False, sep=';', decimal=',')

#print(f'✅ Histórico salvo : {arq_historico}')
#print(f'✅ Resumo salvo    : {arq_resumo}')